# Synthetic data profiling

This deterministic notebook uses production generation, validation, identity, and feature-preparation modules. It does not download data or write to the database.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "backend").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from backend.data_engine.contracts import CompanySize, HealthStatus, Sector, SyntheticCompanyConfig
from backend.data_engine.features import prepare_features
from backend.data_engine.ingest.synthetic import generate_company
from backend.data_engine.validate.identities import check_all_periods
from backend.data_engine.validate.schema_checks import coverage_report, validate_company


In [ ]:
configs = [
    SyntheticCompanyConfig(seed=seed, sector=sector, size=size, health=health)
    for seed, sector, size, health in [
        (1001, Sector.MANUFACTURING, CompanySize.MEDIUM, HealthStatus.HEALTHY),
        (1002, Sector.MANUFACTURING, CompanySize.MEDIUM, HealthStatus.STRESSED),
        (1003, Sector.RETAIL, CompanySize.SMALL, HealthStatus.STABLE),
        (1004, Sector.SERVICES_SAAS, CompanySize.LARGE, HealthStatus.STABLE),
    ]
]
companies = [generate_company(config) for config in configs]
frames = []
for company in companies:
    frame, metadata = prepare_features(company)
    frame["company"] = company.profile.name
    frame["sector"] = company.profile.sector.value
    frame["size"] = company.generator_config.size.value
    frame["health"] = company.generator_config.health.value
    frames.append(frame)
features = pd.concat(frames, ignore_index=True)
features.head()


## Coverage and missingness
Coverage is reported across all periods; missing values remain missing in the raw frame.


In [ ]:
reports = []
for company in companies:
    report = coverage_report(company.periods, mode="all")
    reports.append({"company": company.profile.name, **report.summary})
coverage = pd.DataFrame(reports)
coverage


In [ ]:
numeric_columns = [c for c in features.select_dtypes(include="number").columns if c not in {"fiscal_year", "quarter"}]
missingness = (features[numeric_columns].isna().mean().sort_values(ascending=False).rename("missing_fraction").to_frame())
missingness.head(15)


## Accounting identities and distributions


In [ ]:
identity_results = {company.profile.name: check_all_periods(company.periods) for company in companies}
identity_summary = pd.DataFrame({"company": list(identity_results), "violations": [sum(map(len, result.values())) for result in identity_results.values()]})
identity_summary


In [ ]:
distribution_summary = features.groupby(["sector", "size", "health"])[["revenue", "ebitda", "cash", "total_assets"]].agg(["count", "mean", "std"]).round(2)
distribution_summary


## Deterministic anomaly labels and visualization
The anomaly-enabled fixture is generated through the same production generator. The plot is optional for headless runs.


In [ ]:
anomaly_config = SyntheticCompanyConfig(seed=1005, inject_anomalies=[{"type": "margin_collapse", "start_month": 14, "duration_months": 4, "magnitude": 0.5}])
anomaly_company = generate_company(anomaly_config)
anomaly_frame, _ = prepare_features(anomaly_company)
anomaly_frame[["period_end", "revenue", "gross_profit", "ebitda"]].head()


In [ ]:
try:
    import matplotlib.pyplot as plt
    ax = anomaly_frame.plot(x="period_end", y=["revenue", "gross_profit", "ebitda"], figsize=(10, 4), title="Synthetic anomaly profile")
    ax.set_ylabel("raw currency units")
    plt.tight_layout()
except ImportError:
    print("matplotlib is not installed; numeric profiling above is complete")
